# 12 — Gate 4: the ensemble runner (spec v0.12 §7, §8, §10; VERSION switch added v0.17; manifest v4 added v0.20)

**VERSION = "v4"** (cell 1) solves the re-registered design, `spec/manifest_v4.csv` (study plan v0.20: five value blocks at 20 % each — core habitat, structural connectivity, climate corridors, carbon, biodiversity; seven scenarios × two refugia realizations = 14 voting formulations; refugia = 1/v with a velocity floor, structural connectivity = the transboundary current squared), into `analyses/y2y/runs_v4/`. What changed against v3.1 — every change keyed on a NEW manifest column, so `VERSION = "v3.1"` still reproduces the prior registration from its own manifest (resumable evidence):

- the stack is `input_data/aligned_stack_v4/` — `pr_refresh_manifest` returns that stack's `manifest.json`, so `pr_setup` ingests the v4 layers; cell 2 asserts the ingested refugia layer is the one the manifest registers (`macrorefugia_path`);
- the refugia layer per formulation is read from `macrorefugia_path` (the SSP245 realization patch is no longer a literal) and recorded per formulation in `formulation_meta.json`;
- the EFG block folder comes from `efg_block_version` (v3 → `iucn_efg_v3`);
- the **unguarded MGA runs on the reference formulation only** (`reference_cell`), one band per gap in `unguarded_probes` (2 / 5 / 10 % → `mga_g02.tif`, `mga_g05.tif`, `mga_g10.tif`, each with its `certificates_gNN.csv`, each resumable on its own tif) — the Claim-A estimand and f(g); every other formulation gets its certified anchor and LP twin here and its members from the guarded sweep (18).

Run order for the v4 record: **11c → 12 → 13 → 15 → 18 → 18b → 18c → 19 → 20 → 21**, each run by Ethan in VS Code (no headless runs). Naming convention: things by what they are, codes in parentheses.

`VERSION = "v3.1"` solves the curated-EFG-block manifest `spec/manifest_v3.1.csv` (12 design formulations) into `runs_v3.1/`; `"v1"` reproduces the as-frozen 2026-08-30 run (`runs/`). No prior record is ever overwritten.

Solves every frozen formulation, serial, **fully resumable** (each artifact skipped when its output exists). Per formulation, into `analyses/y2y/runs<_version>/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `kbest/` | engine run, k-best pool (v1 only; the E5 by-product, discharged) | Gurobi binary, opt_gap 1e-4, portfolio 50 @ 5% |
| `twin/` | engine run, LP twin | Gurobi **proportion** (the v0.10 twin ruling) |
| `anchor.tif` + `formulation_meta.json` | certified anchor (every formulation) | `mga_core`, opt_gap from the manifest |
| `mga_g05.tif` (+ `mga_g02.tif`, `mga_g10.tif` under v4) | 50 unguarded MGA members per band (v4: the reference formulation only) | `mga_core`, k=50 |

ssp245 formulations solve on the 245 macrorefugia realization (layer path patched before ingest;
recorded in each `formulation_meta.json`). v1 reference formulation: anchor/MGA exist from Gate 2b; kbest/twin
are manifest pointers to the Gate-2 record. ~45 min per formulation with an unguarded MGA, a few minutes without ⇒
v4 ≈ 14 anchors + twins plus three bands on the reference cell; **live internet throughout** (WLS). Kernel `R (y2y)`.

In [ ]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)   # the ACTIVE version's stack manifest (aligned_stack_v4/manifest.json under v4)

# ---- VERSION switch (study plan v0.17; supersede, never delete) --------------------------------------------
# "v1" = the as-frozen 2026-08-30 record (spec/manifest.csv, runs/); "v3.1" = the curated EFG block with window-derived
# targets (spec/manifest_v3.1.csv, runs_v3.1/; 12 design formulations); "v4" = the re-registered design
# (spec/manifest_v4.csv, runs_v4/; 14 formulations). config.Y2Y_VERSION must agree (the ingested stack is asserted
# against the manifest in cell 2). Every v4 behaviour is keyed on a manifest COLUMN, never on the version string, so an
# earlier manifest reproduces its own record unchanged.
VERSION <- "v4"      # v4 = manifest v4, study plan v0.20 -- five blocks, seven scenarios, floored refugia + squared transboundary current; v3.1 = the prior registration
MANIFEST_REL <- if (VERSION == "v1") "analyses/y2y/spec/manifest.csv" else sprintf("analyses/y2y/spec/manifest_%s.csv", VERSION)
FREEZE_REL   <- if (VERSION == "v1") "analyses/y2y/spec/manifest_freeze.sha256" else sprintf("analyses/y2y/spec/manifest_%s.sha256", VERSION)
RUNS_REL     <- if (VERSION == "v1") "analyses/y2y/runs" else sprintf("analyses/y2y/runs_%s", VERSION)
MAN <- read.csv(file.path(PROJ, MANIFEST_REL), stringsAsFactors = FALSE)
for (col in c("kbest_ref", "twin_ref")) {           # v3+ has no pool/twin pointers: an all-empty column reads as NA
  if (!col %in% names(MAN)) MAN[[col]] <- ""
  MAN[[col]] <- ifelse(is.na(MAN[[col]]), "", as.character(MAN[[col]]))
}
stopifnot(nrow(MAN) %in% c(12, 14))
# the EFG block folder: the manifest's efg_block_version when it carries one (v3 -> iucn_efg_v3; shared by v3, v3.1 and v4),
# else the pre-column rule (v1 -> iucn_efg; minor versions share the block)
EFG_SUBDIR_EXPECTED <- if ("efg_block_version" %in% names(MAN)) {
  stopifnot("formulations disagree on efg_block_version -- STOP" = length(unique(MAN$efg_block_version)) == 1)
  paste0("iucn_efg_", MAN$efg_block_version[1])
} else if (VERSION == "v1") "iucn_efg" else paste0("iucn_efg_", sub("\\..*$", "", VERSION))
# verify the freeze hash before solving against it
dig <- strsplit(readLines(file.path(PROJ, FREEZE_REL))[1], "  ")[[1]][1]
stopifnot("manifest does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(PROJ, MANIFEST_REL))[[1]]), dig))
cat(sprintf("VERSION %s: %s verified against freeze hash %s... (%d formulations)\n", VERSION, basename(MANIFEST_REL), substr(dig, 1, 16), nrow(MAN)))
RUNS <- file.path(PROJ, RUNS_REL)
DO_KBEST <- VERSION == "v1"        # v3+ re-solves anchors, MGA and twins only (study plan v0.17)

# ---- the refugia layer per formulation -----------------------------------------------------------------------
# manifest v4 registers it per row (column macrorefugia_path: the floored 1/v layer of the v4 stack for ssp585, its 245
# realization for ssp245); earlier manifests carry no column, so the pre-v4 literals stand in.
REFUGIA_585_FALLBACK <- "input_data/aligned_stack/climate_type_macrorefugia.tif"
REFUGIA_245_FALLBACK <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"
refugia_path_for <- function(row) {
  p <- if ("macrorefugia_path" %in% names(row)) trimws(as.character(row$macrorefugia_path)) else NA_character_
  if (length(p) == 1 && !is.na(p) && nzchar(p)) return(p)
  if (grepl("^ssp245", row$climate_level)) REFUGIA_245_FALLBACK else REFUGIA_585_FALLBACK
}
i245 <- which(grepl("^ssp245", MAN$climate_level))
paths245 <- unique(vapply(i245, function(i) refugia_path_for(MAN[i, ]), character(1)))
stopifnot("ssp245 formulations disagree on their refugia realization -- STOP" = length(paths245) <= 1)
REFUGIA_245_PATH <- if (length(paths245) == 1) paths245 else REFUGIA_245_FALLBACK   # patched into the 245 base context
cat(sprintf("refugia (ssp245, %d formulations): %s\n", length(i245), REFUGIA_245_PATH))

# ---- the unguarded-MGA policy --------------------------------------------------------------------------------
# manifest v4 (column reference_cell): the unguarded MGA (the Claim-A estimand and f(g)) runs on the reference cell only,
# one band per gap in its unguarded_probes (2 / 5 / 10 % -> mga_g02 / mga_g05 / mga_g10); every other formulation gets
# its members from the guarded sweep (18). Earlier manifests: every formulation, one band at its band_gap_g (mga_g05).
HAS_REFERENCE_CELL <- "reference_cell" %in% names(MAN)
as_flag <- function(x) isTRUE(as.logical(toupper(trimws(as.character(x)))))
is_reference <- function(row) HAS_REFERENCE_CELL && as_flag(row$reference_cell)
mga_probes_for <- function(row) {
  if (!HAS_REFERENCE_CELL) return(as.numeric(row$band_gap_g))
  if (!is_reference(row)) return(numeric(0))
  p <- if ("unguarded_probes" %in% names(row)) trimws(as.character(row$unguarded_probes)) else NA_character_
  if (is.na(p) || !nzchar(p)) return(as.numeric(row$band_gap_g))
  sort(as.numeric(unlist(jsonlite::fromJSON(p))))
}
gap_tag <- function(g) sprintf("g%02d", as.integer(round(100 * g)))     # 0.02 -> g02, 0.05 -> g05, 0.10 -> g10
if (HAS_REFERENCE_CELL) {
  ref_ids <- MAN$formulation_id[vapply(seq_len(nrow(MAN)), function(i) is_reference(MAN[i, ]), logical(1))]
  stopifnot("manifest carries reference_cell but marks no formulation as the reference -- STOP" = length(ref_ids) >= 1)
  for (id in ref_ids)
    cat(sprintf("unguarded MGA (v4 policy): %s only, bands %s; guarded members for every formulation come from 18\n", id,
                paste(sprintf("%g%%", 100 * mga_probes_for(MAN[MAN$formulation_id == id, ])), collapse = " / ")))
} else cat("unguarded MGA (pre-v4 policy): every formulation, one band at its band_gap_g\n")


In [ ]:
# ---- two ingested base contexts, built ONCE (one per climate level) ------------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))

ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REFUGIA_245_PATH   # the realization the manifest registers (cell 1)
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
efg_used <- ctx585$layers$path[ctx585$layers$role == "feature_efg"]
stopifnot("manifest.json enumerates a different EFG block than VERSION expects -- check config.EFG_SUBDIR" =
            length(efg_used) > 0 && all(grepl(paste0("/", EFG_SUBDIR_EXPECTED, "/"), efg_used)))
# the 585 context ingests whatever stack config.Y2Y_VERSION points at (aligned_stack_v4/ under v4): its refugia layer must
# be the one the manifest registers for the ssp585 formulations, else VERSION and config disagree
refugia_ingested <- ctx585$layers$path[ctx585$layers$name == "climate_type_macrorefugia"]
paths585 <- unique(vapply(which(!grepl("^ssp245", MAN$climate_level)), function(i) refugia_path_for(MAN[i, ]), character(1)))
same_file <- function(a, b) {
  abs_of <- function(p) normalizePath(if (grepl("^/", p)) p else file.path(PROJ, p), mustWork = FALSE)
  identical(abs_of(a), abs_of(b))
}
stopifnot("the ingested refugia layer is not the one the manifest registers for ssp585 -- config.Y2Y_VERSION and VERSION disagree; STOP" =
            length(paths585) == 1 && length(refugia_ingested) == 1 && same_file(paths585, refugia_ingested))
cat(sprintf("EFG block: %d features from %s/\n", length(efg_used), EFG_SUBDIR_EXPECTED))
cat(sprintf("refugia: ssp585 %s | ssp245 %s\n", refugia_ingested, REFUGIA_245_PATH))
cat("base contexts ready (585 canonical; 245 with the realization layer patched)\n")

base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585

In [ ]:
# ---- DRY PLAN (no solves): the per-formulation worklist -------------------------------------------
mga_plan <- function(row) {
  cd <- file.path(RUNS, row$formulation_id)
  probes <- mga_probes_for(row)
  if (!length(probes)) return("skip")                                        # v4: not the reference cell
  st <- vapply(probes, function(g) if (file.exists(file.path(cd, sprintf("mga_%s.tif", gap_tag(g))))) "done" else "TODO", character(1))
  if (!HAS_REFERENCE_CELL) return(st[1])                                      # pre-v4: one band per formulation
  paste0("ref ", paste(sprintf("g=%g:%s", 100 * probes, st), collapse = " "))
}
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cd <- file.path(RUNS, row$formulation_id)
  kb <- if (!DO_KBEST) "skip" else if (nzchar(row$kbest_ref)) "ref" else if (file.exists(file.path(cd, "kbest/run_summary.json"))) "done" else "TODO"
  tw <- if (nzchar(row$twin_ref))  "ref" else if (file.exists(file.path(cd, "twin/run_summary.json")))  "done" else "TODO"
  an <- if (file.exists(file.path(cd, "anchor.tif")) && file.exists(file.path(cd, "formulation_meta.json"))) "done" else "TODO"
  cat(sprintf("%-22s %-9s kbest:%-5s twin:%-5s anchor:%-5s mga:%s\n",
              row$formulation_id, sub("_2071_2100", "", row$climate_level), kb, tw, an, mga_plan(row)))
}

In [ ]:
# ---- runner helpers ------------------------------------------------------------------------
form_wt <- function(row) list(w = jsonlite::fromJSON(row$weight_vector),
                              t = jsonlite::fromJSON(row$target_vector))

run_engine_artifact <- function(row, artifact, ov) {
  cd_rel <- file.path(RUNS_REL, row$formulation_id)
  done <- file.path(PROJ, cd_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s exists -- skipped\n", row$formulation_id, artifact)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- do.call(pr_override, c(list(base_for(row),
      targets                    = wt$t,
      feature_weight_multipliers = wt$w,
      results_dir                = cd_rel,
      results_subdir             = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}

# the certified anchor for EVERY formulation (anchor.tif + formulation_meta.json), then the unguarded MGA members for the
# bands mga_probes_for(row) asks for (pre-v4: band_gap_g on every formulation; v4: the reference cell's probes only, the
# rest skipped). Resumable per artifact: nothing is solved when the anchor and every owed band exist; when a band is still
# owed on a formulation whose anchor is already frozen, the anchor is re-solved for the compiled model + x and asserted
# <= 1e-3 relative from formulation_meta.json (the same rule the guarded sweep applies) -- the frozen record STANDS.
run_anchor_and_mga <- function(row) {
  cd <- file.path(RUNS, row$formulation_id)
  probes <- mga_probes_for(row)
  tifs <- file.path(cd, sprintf("mga_%s.tif", gap_tag(probes)))
  todo <- probes[!file.exists(tifs)]
  anchor_done <- file.exists(file.path(cd, "anchor.tif")) && file.exists(file.path(cd, "formulation_meta.json"))
  skip_msg <- "   mga   -> skipped (v4: unguarded MGA on the reference cell only; guarded members come from 18)\n"
  if (anchor_done && !length(todo)) {
    cat(sprintf("   %s/anchor exists%s -- skipped\n", row$formulation_id,
                if (length(probes)) sprintf(" + mga %s", paste(gap_tag(probes), collapse = "/")) else ""))
    if (!length(probes)) cat(skip_msg)
    return(invisible(NULL))
  }
  wt <- form_wt(row)
  actx <- pr_override(base_for(row),
      targets = wt$t, feature_weight_multipliers = wt$w,
      results_dir = file.path(RUNS_REL, row$formulation_id),
      results_subdir = "mga_build",
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  if (anchor_done) {
    meta <- jsonlite::read_json(file.path(cd, "formulation_meta.json"))
    rel <- abs(anchor$z - meta$anchor_objective) / abs(meta$anchor_objective)
    stopifnot("re-solved anchor drifted > 1e-3 relative from formulation_meta.json -- STOP" = rel <= 1e-3)
    a_frozen <- terra::values(terra::rast(file.path(cd, "anchor.tif")))[cm$pu_index] == 1
    cat(sprintf("   anchor re-solved %.6f vs frozen %.6f (rel %.1e) | %d cells differ from anchor.tif (near-ties)\n",
                anchor$z, meta$anchor_objective, rel, sum(a_frozen != anchor$x)))
  } else {
    dir.create(cd, recursive = TRUE, showWarnings = FALSE)
    r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
    v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
    terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U",
                       NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
    jsonlite::write_json(list(
      formulation_id = row$formulation_id, estimator = row$estimator, verdict_rule = row$verdict_rule,
      manifest_version = VERSION, reference_cell = is_reference(row),
      anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
      anchor_runtime_s = anchor$runtime,
      macrorefugia_path = refugia_path_for(row),           # the per-formulation layer (manifest column, or the pre-v4 literal)
      weight_vector = wt$w, target_vector = wt$t,
      k = row$k_requested, g = row$band_gap_g, unguarded_probes = I(probes),
      created_utc = format(Sys.time(), tz = "UTC")),
      file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
    cat(sprintf("   anchor %.6f (gap %.1e, %.0f s) -> anchor.tif + formulation_meta.json\n",
                anchor$z, anchor$gap, anchor$runtime))
  }
  if (!length(probes)) { cat(skip_msg); return(invisible(NULL)) }
  for (g in todo) {
    cat(sprintf("   mga   -> band g = %g%% (%s), k = %d\n", 100 * g, gap_tag(g), row$k_requested))
    gen <- mga_generate(cm, anchor, g = g, k = row$k_requested)
    mga_write(gen, cm, actx$cost, cd, gap_tag(g))
  }
  invisible(NULL)
}

In [ ]:
# ---- THE LOOP: serial over the frozen formulations (resumable anywhere) --------------------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  if (!DO_KBEST) {
    cat("   kbest -> skipped (v3+: anchors, MGA, twins only)\n")
  } else if (!nzchar(row$kbest_ref)) {
    run_engine_artifact(row, "kbest", list(solver = "gurobi", decision_type = "binary",
                                           opt_gap = row$opt_gap, portfolio_n = row$k_requested,
                                           portfolio_gap = row$band_gap_g))
  } else cat(sprintf("   kbest -> %s (Gate-2 record)\n", row$kbest_ref))
  if (!nzchar(row$twin_ref)) {
    run_engine_artifact(row, "twin", list(solver = "gurobi", decision_type = "proportion",
                                          opt_gap = row$opt_gap, portfolio_n = 1))
  } else cat(sprintf("   twin  -> %s (Gate-2 record, HiGHS exact)\n", row$twin_ref))
  # v1 reference formulation: Gate 2b wrote gate2b_meta.json before the naming settled --
  # derive the standard meta file once so 13 reads every formulation uniformly
  cd <- file.path(RUNS, row$formulation_id)
  g2b <- file.path(cd, "gate2b_meta.json")
  fmeta <- file.path(cd, "formulation_meta.json")
  if (!file.exists(fmeta) && file.exists(g2b)) {
    m <- jsonlite::read_json(g2b)
    m$formulation_id <- row$formulation_id
    jsonlite::write_json(m, fmeta, auto_unbox = TRUE, pretty = TRUE, digits = 10)
    cat("   formulation_meta.json derived from gate2b_meta.json\n")
  }
  run_anchor_and_mga(row)     # anchor for every formulation; unguarded bands per the manifest's policy
  cat(sprintf("== %s done | batch elapsed %.1f h\n", row$formulation_id,
              (proc.time()[["elapsed"]] - t_batch) / 3600))
}
cat("\nENSEMBLE COMPLETE -- next: analyses/y2y/13_gate4_analysis.ipynb\n")

In [ ]:
# ---- timing + integrity summary ------------------------------------------------------------
# an unguarded band is required only where the manifest's policy asks for one (pre-v4: mga_g05 on every formulation;
# v4: the reference cell's probes) -- a non-reference v4 formulation is complete with its anchor, meta and twin
mga_status <- function(row, cd) {
  probes <- mga_probes_for(row)
  if (!length(probes)) return("mga skipped (reference cell only)")
  parts <- vapply(probes, function(g) {
    f <- file.path(cd, sprintf("certificates_%s.csv", gap_tag(g)))
    if (!file.exists(f)) return(sprintf("%s MISSING", gap_tag(g)))
    ce <- read.csv(f)
    sprintf("%s %d members%s %.0f min", gap_tag(g), nrow(ce), if (all(ce$band_ok)) "" else " BAND-VIOLATED", sum(ce$runtime_s) / 60)
  }, character(1))
  paste0("mga ", paste(parts, collapse = ", "))
}
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  meta_f <- file.path(cd, "formulation_meta.json")
  kb_f <- if (nzchar(row$kbest_ref)) file.path(PROJ, row$kbest_ref, "run_summary.json")
          else file.path(cd, "kbest/run_summary.json")
  tw_f <- if (nzchar(row$twin_ref)) file.path(PROJ, row$twin_ref, "run_summary.json")
          else file.path(cd, "twin/run_summary.json")
  if (!file.exists(meta_f) || (DO_KBEST && !file.exists(kb_f)) || !file.exists(tw_f)) {
    cat(sprintf("%-22s INCOMPLETE\n", row$formulation_id)); next }
  m <- jsonlite::read_json(meta_f); tw <- jsonlite::read_json(tw_f)
  kb <- if (DO_KBEST) jsonlite::read_json(kb_f) else list(n_alternatives = NA_integer_, solve_seconds = NA_real_)
  tw_obj <- tryCatch(as.numeric(unlist(tw$solver_provenance$objective))[1], error = function(e) NA)
  ok <- if (is.na(tw_obj)) "?" else if (tw_obj <= m$anchor_objective + 1e-6) "OK" else "VIOLATED"
  cat(sprintf("%-22s anchor %.6f (gap %.0e, %4.0fs) | twin %.6f [LP<=MILP %s] | kbest %d sol %5.0fs | %s\n",
              row$formulation_id, m$anchor_objective, m$anchor_gap, m$anchor_runtime_s,
              ifelse(is.na(tw_obj), NaN, tw_obj), ok, kb$n_alternatives, kb$solve_seconds, mga_status(row, cd)))
}

In [8]:
# ---- OPTIONAL: HiGHS spot-check twin (2nd cell; reference already has one) -----------------
# Flip to TRUE and run overnight if desired (worst observed HiGHS case: 109 min).
RUN_HIGHS_SPOTCHECK <- FALSE
if (RUN_HIGHS_SPOTCHECK) {
  row <- MAN[MAN$formulation_id == "s4_ssp585_theta3", ]
  run_engine_artifact(row, "twin_highs", list(solver = "highs", decision_type = "proportion",
                                              portfolio_n = 1))
}